In [1]:
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from PIL import Image
import torch

/Users/naveen1.mathur/Desktop/x/sftc/_learning/ml/mlp/ml-projects/_env_05_OCR_DOCUMENT_PARSER/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_path = "./files/invoice.pdf"
pages = convert_from_path(pdf_path, dpi=1000)
image = pages[0]
image.save("./files/page1.png", "PNG")

/Users/naveen1.mathur/Desktop/x/sftc/_learning/ml/mlp/ml-projects/_env_05_OCR_DOCUMENT_PARSER/lib/python3.9/site-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (94815000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


In [3]:
ocr_data = pytesseract.image_to_data(image, output_type=Output.DICT)

words = []
boxes = []

W, H = image.size  # width, height of page

In [4]:
for i in range(len(ocr_data["text"])):
    if ocr_data["text"][i].strip() != "":
        words.append(ocr_data["text"][i])

        # Tesseract gives x, y, w, h
        x, y, w, h = (
            ocr_data["left"][i],
            ocr_data["top"][i],
            ocr_data["width"][i],
            ocr_data["height"][i],
        )

        # Normalize bbox to 0–1000 as expected by LayoutLMv3
        boxes.append(
            [
                int(1000 * x / W),
                int(1000 * y / H),
                int(1000 * (x + w) / W),
                int(1000 * (y + h) / H),
            ]
        )

In [5]:
processor = LayoutLMv3Processor.from_pretrained(
    "nielsr/layoutlmv3-finetuned-cord", apply_ocr=False
)
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "nielsr/layoutlmv3-finetuned-cord"
)

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 16384.00it/s]
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RobertaTokenizer'. 
The class this function is called from is 'LayoutLMv3TokenizerFast'.


In [6]:
model.to("mps")

LayoutLMv3ForTokenClassification(
  (layoutlmv3): LayoutLMv3Model(
    (embeddings): LayoutLMv3TextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (x_position_embeddings): Embedding(1024, 128)
      (y_position_embeddings): Embedding(1024, 128)
      (h_position_embeddings): Embedding(1024, 128)
      (w_position_embeddings): Embedding(1024, 128)
    )
    (patch_embed): LayoutLMv3PatchEmbeddings(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
    (encoder): LayoutLMv3Encoder

In [7]:
# encoding = processor(
#     image,
#     words,
#     boxes=boxes,
#     return_tensors="pt",
#     truncation=True,
#     padding="max_length",
# )

encoding = processor(
    image,
    words,
    boxes=boxes,
    return_tensors="pt",
    truncation=True,
    stride=128,  # overlap context
    padding="max_length",
)

In [8]:
for k, v in encoding.items():
    encoding[k] = v.to("mps")

In [9]:
with torch.no_grad():
    outputs = model(**encoding)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()

/Users/naveen1.mathur/Desktop/x/sftc/_learning/ml/mlp/ml-projects/_env_05_OCR_DOCUMENT_PARSER/lib/python3.9/site-packages/transformers/modeling_utils.py:1742: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [10]:
labels = [model.config.id2label[p] for p in predictions]

In [11]:
OCR_TEXT = ""
for word, label in zip(words, labels):
    if label != "O":
        print(f"{label:15} -> {word}")
        OCR_TEXT += f"{word} "

B-MENU.PRICE    -> Reliance
B-MENU.NM       -> Retail
I-MENU.SUB_NM   -> Limited
I-MENU.SUB_NM   -> SCO
I-MENU.SUB_NM   -> No.123
B-MENU.NUM      -> -
B-MENU.NUM      -> 124,
B-MENU.NUM      -> Sector
B-MENU.NUM      -> -
B-MENU.NUM      -> 17C
B-MENU.CNT      -> Chandigarh
B-MENU.UNITPRICE -> Chandigarh
I-MENU.CNT      -> Chandigarh
B-MENU.NUM      -> 160017
B-MENU.CNT      -> (Original
B-MENU.NUM      -> for
I-MENU.CNT      -> Recipient)
B-MENU.PRICE    -> Tax
I-MENU.SUB_NM   -> Invoice
I-MENU.SUB_NM   -> Invoice
I-MENU.SUB_NM   -> No
B-MENU.PRICE    -> :
I-MENU.SUB_NM   -> A4R26R9999256622
I-MENU.SUB_NM   -> Invoice/Payment
I-MENU.SUB_NM   -> Date
B-MENU.NM       -> &
I-MENU.SUB_NM   -> Time
I-MENU.SUB_NM   -> :
I-MENU.SUB_NM   -> 06
B-MENU.NUM      -> Aug,2025
B-MENU.NUM      -> 20:39:27
B-MENU.SUB_NM   -> PAN
I-MENU.SUB_NM   -> No
B-MENU.DISCOUNTPRICE -> :
B-MENU.SUB_NM   -> AABCR1718E
I-MENU.SUB_NM   -> GST
I-MENU.SUB_NM   -> No
B-SUB_TOTAL.DISCOUNT_PRICE -> :
B-SUB_TOTAL.DISCOUN

In [12]:
import torch

if torch.backends.mps.is_available():
    print("Using MPS (Apple GPU)")
print(torch.mps.current_allocated_memory() / 1024**2, "MB allocated")
print(torch.mps.driver_allocated_memory() / 1024**2, "MB reserved by driver")

Using MPS (Apple GPU)
481.3681640625 MB allocated
1046.671875 MB reserved by driver


In [13]:
from llama_cpp import Llama

In [14]:
# model_path = "./llms/mistral-7b-instruct-v0.2.Q8_0.gguf"
model_path = "./files/mistral-7b-instruct-v0.2.Q8_0.gguf"
MAX_PROMPT_TOKENS = 2**13
MAX_OUTPUT_TOKENS = 2**12

mistral_ggufQ8_0 = Llama(
    model_path=model_path,
    n_ctx=MAX_PROMPT_TOKENS,
    use_mlock=True,
    n_threads=8,
    n_gpu_layers=16,
)

llama_model_load_from_file_impl: using device Metal (Apple M1 Pro) - 9876 MiB free
llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from ./files/mistral-7b-instruct-v0.2.Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.ro

In [15]:
def promptGeneration00(text: str) -> str:
    return f"""
    ### Instruction:
    You are an AI assistant. Analyze the following text and generate structured answers in **valid JSON format only**.
    - Do not include explanations.
    - Do not include extra text outside JSON.
    - Ensure the JSON is syntactically correct.

    ### Text:
    {text}

    ### Response (JSON only):
    """

In [16]:
def promptGeneration(text):
    return f"""
        ### Instruction:
        You are an AI assistant. Analyze the following text and generate structured answers in **valid JSON format only**.
        - find customer_name and return it under key 'a'
        - find gst_number and return it under key 'b'
        - find bill_product_names and return it under key 'e'
        - find product amount and include taxes and return it under key 'c'
        - find invoice date and return it under key 'f'
        - find date of issue of invoice / bill date and return it under key 'g'
        - Do not include explanations.
        - Do not include extra text outside JSON.
        - Ensure the JSON is syntactically correct.

        Do not include any other keys.
        Use keys specified in brackets for giving these output.
        If a field is missing, set its value to null.
        Output strictly in compact JSON within max token limits,

        ### Text:
        {text}

        ### Response:
        """

In [17]:
def llmOutput(prompt, output_tokens=MAX_OUTPUT_TOKENS):
    modelOutput = mistral_ggufQ8_0(
        prompt=prompt,
        max_tokens=output_tokens,
        temperature=0.7,  # randomness (0 = deterministic, >1 = creative).
        # top_k=128, # sample only from top-K tokens
        top_p=0.9,
        repeat_penalty=1.05,  # if model repeats a lot, increase
        stop=["###"],
    )
    modelResponse = modelOutput["choices"][0]["text"].strip()
    return modelResponse

In [18]:
# prompt = promptGeneration00(text=OCR_TEXT)
prompt = promptGeneration(text=OCR_TEXT)
output = llmOutput(prompt=prompt)

llama_perf_context_print:        load time =   11553.93 ms
llama_perf_context_print: prompt eval time =   11552.34 ms /   697 tokens (   16.57 ms per token,    60.33 tokens per second)
llama_perf_context_print:        eval time =   15773.67 ms /   146 runs   (  108.04 ms per token,     9.26 tokens per second)
llama_perf_context_print:       total time =   27388.29 ms /   843 tokens
llama_perf_context_print:    graphs reused =        140


In [19]:
print(output)

{
            "a": "Devka Trehan",
            "b": "O4AABCR1718E1ZX",
            "c": 1412.46,
            "e": ["Telecommunication services", "Platform services"],
            "f": "06 Aug,2025"
        }
        {
            "a": null,
            "b": null,
            "c": null,
            "e": ["Tax Invoice"],
            "f": "06 Aug,2025",
            "g": "06 Aug,2025"
        }


In [24]:
# full_pipeline_layoutlmv3.py

import pdfplumber
from PIL import Image
from datasets import Dataset
import torch
from transformers import (
    AutoProcessor,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
)

In [25]:
# -----------------------------
# 1. PDF → dataset
# -----------------------------
pdf_path = "./files/invoice.pdf"
dataset_list = []

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        # Convert page to PIL image
        img = page.to_image(resolution=300).original

        # Extract words + bbox
        words = []
        bboxes = []
        for word in page.extract_words():
            words.append(word["text"])
            # Normalize bbox to 0-1000
            x0 = int(word["x0"] / page.width * 1000)
            y0 = int(word["top"] / page.height * 1000)
            x1 = int(word["x1"] / page.width * 1000)
            y1 = int(word["bottom"] / page.height * 1000)
            bboxes.append([x0, y0, x1, y1])

        # Labels (0=O for unsupervised; replace with manual annotations for NER)
        labels = [0] * len(words)

        dataset_list.append(
            {"image": img, "words": words, "bboxes": bboxes, "labels": labels}
        )

dataset = Dataset.from_list(dataset_list)

In [ ]:
processor = LayoutLMv3Processor.from_pretrained(
    "nielsr/layoutlmv3-finetuned-cord", apply_ocr=False
)
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "nielsr/layoutlmv3-finetuned-cord"
)

In [29]:
# -----------------------------
# 2. Processor + Preprocessing
# -----------------------------
model_name = "microsoft/layoutlmv3-base"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    apply_ocr=False,
)  # adjust num_labels


def preprocess(batch):
    encoded = processor(
        images=batch["image"],
        text=batch["words"],
        boxes=batch["bboxes"],
        word_labels=batch["labels"],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
    )
    return encoded


encoded_dataset = dataset.map(
    preprocess, batched=True, remove_columns=dataset.column_names
)

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 2462.89it/s]


TypeError: __init__() got an unexpected keyword argument 'apply_ocr'

In [ ]:
# -----------------------------
# 3. Training setup
# -----------------------------
args = TrainingArguments(
    output_dir="./layoutlmv3-finetune",
    per_device_train_batch_size=1,  # for single PDF
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    logging_steps=1,
    save_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_dataset,
)

In [ ]:
trainer.train()

In [ ]:
# -----------------------------
# 5. Save model + processor
# -----------------------------
model.save_pretrained("./layoutlmv3-finetune")
processor.save_pretrained("./layoutlmv3-finetune")